# IncludeHer UK - Interactive Data Explorer
our project exploring gender representation in the UK science specifications for 14-18 year olds.

Explore gender representation, mention types, subjects, and regional demographics in UK science exam specifications.

**How to use**
- Use the dropdown menus to choose a key stage, exam board, and chart view. The explorer opens on KS5 with all exam boards pooled.
- Choose **All exam boards** to look at every specification in a key stage at once (the same person is counted once). Choose **Both — compare KS4 and KS5** to put the two stages side by side.
- Hover over chart segments for counts and percentages.
- Click a segment (or pick a demographic below) to list every named scientist in that group.


In [1]:
import contextlib
import io
import os
import sys
from collections import Counter

import ipywidgets as widgets
from IPython.display import HTML, display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, os.path.abspath("."))
import Plot_Figures_UK as pf

COLOURS = pf.COLOURS
MALE_COLOUR = COLOURS[1]
FEMALE_COLOUR = COLOURS[0]
CONCEPT_COLOUR = COLOURS[3]
SCIENTIST_COLOUR = COLOURS[2]
REGION_COLOURS = pf.REGION_DONUT_COLOURS

ALL_BOARDS = "All exam boards"
STAGE_BOTH = "both"

KEY_STAGE_OPTIONS = {
    "KS5 - A-Level / SQ Adv + Highers (ages 16-18)": "ks5",
    "KS4 - GCSE / SQ5 (ages 14-16)": "ks4",
    "Both - compare KS4 and KS5": STAGE_BOTH,
}

VIEW_OPTIONS = [
    "Subject breakdown (by gender)",
    "Unique scientists by gender",
    "Concept vs scientist mentions",
    "Scientists by region",
    "Scientists by nationality",
]

COMPARE_BOARD_MAP = {
    "AQA": {"ks4": ["AQA"], "ks5": ["AQA"]},
    "CCEA": {"ks4": ["CCEA"], "ks5": ["CCEA"]},
    "Edexcel": {"ks4": ["Edexcel"], "ks5": ["Edexcel"]},
    "OCR": {"ks4": ["OCR A", "OCR B"], "ks5": ["OCR"]},
    "Scottish": {"ks4": ["Scottish NQ5"], "ks5": ["Scottish highers"]},
    "WJEC": {"ks4": ["WJEC"], "ks5": ["WJEC"]},
}
COMPARE_BOARD_ORDER = ["AQA", "CCEA", "Edexcel", "OCR", "Scottish", "WJEC"]
COMPARE_BOARD_LABELS = {
    "OCR": "Cambridge OCR",
    "Scottish": "SQ",
}

SUBJECT_ORDER = [
    "physics",
    "chemistry",
    "biology",
    "environmental science",
    "geology",
    "astronomy",
]

DEMOGRAPHIC_OPTIONS = {
    "Gender": "gender",
    "Region": "region",
    "Nationality": "nationality",
}

SUBJECT_LABELS = {
    "environmental science": "Environmental science",
    "biology": "Biology",
    "physics": "Physics",
    "chemistry": "Chemistry",
    "geology": "Geology",
    "astronomy": "Astronomy",
}

with contextlib.redirect_stdout(io.StringIO()):
    DATA = {
        "ks5": pf.prepare_key_stage("ks5"),
        "ks4": pf.prepare_key_stage("ks4"),
    }

def _title_case_name(name):
    return " ".join(word.capitalize() for word in str(name).split())


def _count_line(label, stats):
    return (
        f"{label}: {stats['total']} unique scientists "
        f"({stats['male']} male, {stats['female']} female)"
    )


def _women_line(label, stats):
    names = ", ".join(_title_case_name(n) for n in stats["women"]) or "(none)"
    return f"{label}: {names}"


summary = pf.unique_scientist_summary(DATA["ks5"]["scientists"], DATA["ks4"]["scientists"])
display(HTML(f"""
<div style="height:4px;width:100%;background:{pf.TEAL};border-radius:2px;margin:4px 0 16px;"></div>
<div style="font-size:15px;line-height:1.75;color:#1a1a1a;">
{_count_line("A-Level / SQ Adv + Highers", summary["ks5"])}<br>
{_count_line("GCSE / SQ5", summary["ks4"])}<br>
{_count_line("KS4 + KS5 combined", summary["combined"])}<br>
{_women_line("KS5 women", summary["ks5"])}<br>
{_women_line("KS4 women", summary["ks4"])}
</div>
"""))


In [2]:
def prettify(text):
    if not text or str(text).lower() in ("nan", "none", "unknown", ""):
        return "Unknown"
    return str(text).strip().title().replace("Eruope", "Europe")


def prettify_name(name):
    if not name:
        return ""
    parts = []
    for chunk in str(name).split(","):
        parts.append(" ".join(word.capitalize() for word in chunk.strip().split()))
    return ", ".join(parts)


def stage_cfg(stage_key):
    return DATA[stage_key]["cfg"]


def stage_dataset(stage_key):
    return DATA[stage_key]["dataset"]


def board_file_key(stage_key, board_display):
    return stage_cfg(stage_key)["boards_map"][board_display]


def board_labels(stage_key):
    return stage_cfg(stage_key)["board_labels"]


def display_board_name(stage_key, board):
    return board_labels(stage_key).get(board, board)


def format_board_option(stage_key, value):
    if value == ALL_BOARDS:
        return ALL_BOARDS
    if stage_key == STAGE_BOTH:
        return COMPARE_BOARD_LABELS.get(value, value)
    return display_board_name(stage_key, value)


def board_options(stage_key):
    if stage_key == STAGE_BOTH:
        return [ALL_BOARDS] + COMPARE_BOARD_ORDER
    return [ALL_BOARDS] + list(stage_cfg(stage_key)["boards_display"])


def resolve_boards(stage_key, board_sel):
    if board_sel in (None, ALL_BOARDS):
        return list(stage_cfg(stage_key)["boards_display"])
    if board_sel in COMPARE_BOARD_MAP:
        return list(COMPARE_BOARD_MAP[board_sel][stage_key])
    return [board_sel]


def scope_label(stage_key, board_sel):
    if board_sel in (None, ALL_BOARDS):
        return "all exam boards"
    if stage_key == STAGE_BOTH:
        extra = {
            "OCR": "Cambridge OCR (KS5 vs A & B at KS4)",
            "Scottish": "SQ (SQ5 vs SQ Adv + Highers)",
        }
        return extra.get(board_sel, board_sel)
    return display_board_name(stage_key, board_sel)


def subjects_for_board(stage_key, board_display):
    dataset = stage_dataset(stage_key)
    board_key = board_file_key(stage_key, board_display)
    core = {"physics", "chemistry", "biology"}
    return [
        sb
        for sb, info in dataset[board_key]["subjects"].items()
        if sb in core
        or (
            "concept" in info
            and info["concept"]["male"] + info["concept"]["female"] > 0
        )
    ]


def subjects_for_boards(stage_key, boards):
    seen = []
    for board in boards:
        for subject in subjects_for_board(stage_key, board):
            if subject not in seen:
                seen.append(subject)
    ordered = [s for s in SUBJECT_ORDER if s in seen]
    return ordered + [s for s in seen if s not in SUBJECT_ORDER]


def merge_scientists(stage_key, board_sel=ALL_BOARDS):
    dataset = stage_dataset(stage_key)
    boards = resolve_boards(stage_key, board_sel)
    merged = {}
    for board in boards:
        board_key = board_file_key(stage_key, board)
        for name, info in dataset[board_key]["names"].items():
            if not pf.is_valid_scientist_name(name):
                continue
            label = display_board_name(stage_key, board)
            if name not in merged:
                merged[name] = {
                    "gender": info.get("gender", "unknown"),
                    "region": info.get("region", "unknown"),
                    "nationality": info.get("nationality", "unknown"),
                    "mentions": info.get("number of mentions", 0),
                    "boards": [label],
                }
            else:
                if label not in merged[name]["boards"]:
                    merged[name]["boards"].append(label)
                merged[name]["mentions"] = max(
                    merged[name]["mentions"], info.get("number of mentions", 0)
                )
    return merged


def merge_both_stages(board_sel=ALL_BOARDS):
    merged = {}
    for stage_key, tag in (("ks4", "KS4"), ("ks5", "KS5")):
        for name, info in merge_scientists(stage_key, board_sel).items():
            boards = [f"{board} ({tag})" for board in info.get("boards") or []]
            if name not in merged:
                merged[name] = {
                    "gender": info.get("gender", "unknown"),
                    "region": info.get("region", "unknown"),
                    "nationality": info.get("nationality", "unknown"),
                    "mentions": info.get("mentions", 0),
                    "boards": boards,
                }
            else:
                for board in boards:
                    if board not in merged[name]["boards"]:
                        merged[name]["boards"].append(board)
                merged[name]["mentions"] = max(
                    merged[name]["mentions"], info.get("mentions", 0)
                )
    return merged


def scientists_for_filters(stage_key, board_sel):
    if stage_key == STAGE_BOTH:
        return merge_both_stages(board_sel)
    return merge_scientists(stage_key, board_sel)


def unique_gender_counts(scientists):
    male = sum(1 for s in scientists.values() if str(s.get("gender", "")).lower() == "male")
    female = sum(1 for s in scientists.values() if str(s.get("gender", "")).lower() == "female")
    return male, female


def subject_mention_totals(stage_key, board_sel):
    boards = resolve_boards(stage_key, board_sel)
    dataset = stage_dataset(stage_key)
    totals = {}
    for board in boards:
        board_key = board_file_key(stage_key, board)
        for subject, info in dataset[board_key]["subjects"].items():
            bucket = totals.setdefault(
                subject,
                {"concept": {"male": 0, "female": 0}, "scientist": {"male": 0, "female": 0}},
            )
            for kind in ("concept", "scientist"):
                cat = info.get(kind, {"male": 0, "female": 0})
                bucket[kind]["male"] += cat.get("male", 0)
                bucket[kind]["female"] += cat.get("female", 0)
    subjects = subjects_for_boards(stage_key, boards)
    return subjects, totals


def mention_type_counts(stage_key, board_sel):
    boards = resolve_boards(stage_key, board_sel)
    dataset = stage_dataset(stage_key)
    concept = 0
    scientist = 0
    for board in boards:
        overall = dataset[board_file_key(stage_key, board)]["overall"]
        concept += overall["concept"]["male"] + overall["concept"]["female"]
        scientist += overall["scientist"]["male"] + overall["scientist"]["female"]
    return concept, scientist


def filter_scientists(scientists, gender=None, region=None, nationality=None):
    rows = []
    for name, info in scientists.items():
        g = str(info.get("gender", "unknown")).lower()
        r = str(info.get("region", "unknown")).lower()
        n = str(info.get("nationality", "unknown")).lower()
        if gender and g != gender.lower():
            continue
        if region and r != region.lower():
            continue
        if nationality and n != nationality.lower():
            continue
        rows.append((name, info))
    return rows


def demographic_values(scientists, dimension):
    counter = Counter(str(info.get(dimension, "unknown")).lower() for info in scientists.values())
    return sorted(counter.keys(), key=lambda k: (-counter[k], k))


def demographic_counts(scientists, dimension):
    counter = Counter(str(info.get(dimension, "unknown")).lower() for info in scientists.values())
    labels = sorted(counter.keys(), key=lambda k: (-counter[k], k))
    counts = [counter[k] for k in labels]
    total = sum(counts) or 1
    display_labels = [prettify(k) for k in labels]
    hover = [
        f"{prettify(k)}<br>Count: {counter[k]}<br>Share: {counter[k] / total * 100:.1f}%"
        for k in labels
    ]
    return labels, counts, display_labels, hover, total


def names_html(rows, title):
    if not rows:
        return f"<p><em>No scientists match <strong>{title}</strong>.</em></p>"
    lines = [
        "<div style='max-height:320px; overflow-y:auto; border:1px solid #ddd; border-radius:8px; padding:12px; background:#fafafa;'>",
        f"<p style='margin-top:0'><strong>{title}</strong> - {len(rows)} scientist{'s' if len(rows) != 1 else ''}</p>",
        "<table style='width:100%; border-collapse:collapse; font-size:14px;'>",
        "<tr style='background:#eef3f8'><th align='left'>Name</th><th align='left'>Gender</th><th align='left'>Region</th><th align='left'>Nationality</th><th align='right'>Mentions</th></tr>",
    ]
    for name, info in sorted(rows, key=lambda x: prettify_name(x[0])):
        lines.append(
            "<tr style='border-top:1px solid #eee'>"
            f"<td>{prettify_name(name)}</td>"
            f"<td>{prettify(info.get('gender'))}</td>"
            f"<td>{prettify(info.get('region'))}</td>"
            f"<td>{prettify(info.get('nationality'))}</td>"
            f"<td align='right'>{info.get('mentions', 0)}</td>"
            "</tr>"
        )
    lines.extend(["</table>", "</div>"])
    return "".join(lines)


def pie_figure(labels, counts, display_labels, hover, title, colours=None):
    colours = colours or REGION_COLOURS
    fig = go.Figure(
        data=[
            go.Pie(
                labels=display_labels,
                values=counts,
                hole=0.45,
                marker=dict(colors=[colours[i % len(colours)] for i in range(len(counts))]),
                textinfo="percent+label",
                textposition="outside",
                hovertext=hover,
                hoverinfo="text",
                customdata=labels,
            )
        ]
    )
    fig.update_layout(title=title, height=480, margin=dict(t=70, b=20, l=20, r=20))
    return fig


def subject_figure(stage_key, board_sel):
    subjects, totals = subject_mention_totals(stage_key, board_sel)
    y_labels = [SUBJECT_LABELS.get(sb, sb.title()) for sb in subjects]
    male_c, female_c, male_s, female_s = [], [], [], []
    hover_c, hover_s = [], []
    for sb in subjects:
        cat_c = totals[sb]["concept"]
        cat_s = totals[sb]["scientist"]
        mc, fc = cat_c.get("male", 0), cat_c.get("female", 0)
        ms, fs = cat_s.get("male", 0), cat_s.get("female", 0)
        male_c.append(mc)
        female_c.append(fc)
        male_s.append(ms)
        female_s.append(fs)
        tot_c = mc + fc or 1
        tot_s = ms + fs or 1
        hover_c.append(
            f"Concept mentions<br>Men: {mc} ({mc / tot_c * 100:.1f}%)<br>Women: {fc} ({fc / tot_c * 100:.1f}%)"
        )
        hover_s.append(
            f"Scientist mentions<br>Men: {ms} ({ms / tot_s * 100:.1f}%)<br>Women: {fs} ({fs / tot_s * 100:.1f}%)"
        )
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Concept mentions", "Scientist mentions"),
        horizontal_spacing=0.12,
    )
    fig.add_trace(
        go.Bar(y=y_labels, x=male_c, name="Men", orientation="h", marker_color=MALE_COLOUR, hovertext=hover_c, hoverinfo="text"),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(y=y_labels, x=female_c, name="Women", orientation="h", marker_color=FEMALE_COLOUR, hovertext=hover_c, hoverinfo="text"),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(y=y_labels, x=male_s, name="Men ", orientation="h", marker_color=MALE_COLOUR, hovertext=hover_s, hoverinfo="text", showlegend=False),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Bar(y=y_labels, x=female_s, name="Women ", orientation="h", marker_color=FEMALE_COLOUR, hovertext=hover_s, hoverinfo="text", showlegend=False),
        row=1,
        col=2,
    )
    fig.update_layout(
        barmode="stack",
        title=f"Subject breakdown - {scope_label(stage_key, board_sel)}",
        height=max(360, 55 * len(subjects) + 120),
        margin=dict(t=80, l=140),
    )
    fig.update_xaxes(title_text="Mentions")
    return fig


def gender_figure(stage_key, board_sel):
    scientists = merge_scientists(stage_key, board_sel)
    male, female = unique_gender_counts(scientists)
    total = male + female or 1
    labels = ["male", "female"]
    counts = [male, female]
    display_labels = ["Men", "Women"]
    hover = [
        f"Men<br>Count: {male}<br>Share: {male / total * 100:.1f}%",
        f"Women<br>Count: {female}<br>Share: {female / total * 100:.1f}%",
    ]
    return pie_figure(
        labels,
        counts,
        display_labels,
        hover,
        f"Unique named scientists by gender - {scope_label(stage_key, board_sel)}",
        colours=[MALE_COLOUR, FEMALE_COLOUR],
    )


def mention_type_figure(stage_key, board_sel):
    concept, scientist = mention_type_counts(stage_key, board_sel)
    total = concept + scientist or 1
    labels = ["concept", "scientist"]
    counts = [concept, scientist]
    display_labels = ["Concept", "Scientist"]
    hover = [
        f"Concept mentions<br>Count: {concept}<br>Share: {concept / total * 100:.1f}%",
        f"Scientist mentions<br>Count: {scientist}<br>Share: {scientist / total * 100:.1f}%",
    ]
    return pie_figure(
        labels,
        counts,
        display_labels,
        hover,
        f"Mention type - {scope_label(stage_key, board_sel)}",
        colours=[CONCEPT_COLOUR, SCIENTIST_COLOUR],
    )


def region_figure(stage_key, board_sel=ALL_BOARDS):
    scientists = merge_scientists(stage_key, board_sel)
    labels, counts, display_labels, hover, _ = demographic_counts(scientists, "region")
    return pie_figure(labels, counts, display_labels, hover, f"Scientists by region - {scope_label(stage_key, board_sel)}")


def nationality_figure(stage_key, board_sel=ALL_BOARDS):
    scientists = merge_scientists(stage_key, board_sel)
    labels, counts, display_labels, hover, total = demographic_counts(scientists, "nationality")
    colours = [REGION_COLOURS[i % len(REGION_COLOURS)] for i in range(len(counts))]
    fig = go.Figure(
        data=[
            go.Bar(
                y=display_labels[::-1],
                x=counts[::-1],
                orientation="h",
                marker_color=colours[::-1],
                hovertext=hover[::-1],
                hoverinfo="text",
                customdata=labels[::-1],
                text=[f"{c} ({c / total * 100:.1f}%)" for c in counts[::-1]],
                textposition="outside",
                cliponaxis=False,
            )
        ]
    )
    fig.update_layout(
        title=f"Scientists by nationality - {scope_label(stage_key, board_sel)}",
        xaxis_title="Unique scientists",
        height=max(420, 28 * len(counts) + 140),
        margin=dict(l=160, r=90, t=70, b=50),
        showlegend=False,
    )
    return fig


def grouped_stage_bars(y_labels, ks4_x, ks5_x, hover_g, hover_a, customdata, title, xaxis_title, height):
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            y=y_labels,
            x=ks4_x,
            name="KS4 (GCSE / SQ5)",
            orientation="h",
            marker_color=pf.PURPLE,
            hovertext=hover_g,
            hoverinfo="text",
            customdata=customdata,
        )
    )
    fig.add_trace(
        go.Bar(
            y=y_labels,
            x=ks5_x,
            name="KS5 (A-Level / SQ Adv + Highers)",
            orientation="h",
            marker_color=pf.PURPLE_LIGHT,
            hovertext=hover_a,
            hoverinfo="text",
            customdata=customdata,
        )
    )
    fig.update_layout(
        barmode="group",
        title=title,
        xaxis_title=xaxis_title,
        height=height,
        margin=dict(l=160, t=70),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    )
    return fig


def subject_comparison_figure(board_sel):
    subjects_g, totals_g = subject_mention_totals("ks4", board_sel)
    subjects_a, totals_a = subject_mention_totals("ks5", board_sel)
    subjects = [s for s in SUBJECT_ORDER if s in set(subjects_g) | set(subjects_a)]
    extras = [s for s in list(dict.fromkeys(subjects_g + subjects_a)) if s not in SUBJECT_ORDER]
    subjects = subjects + extras
    y_labels = [SUBJECT_LABELS.get(sb, sb.title()) for sb in subjects]

    def gender_split(totals, subject, kind):
        cat = totals.get(subject, {}).get(kind, {"male": 0, "female": 0})
        return cat.get("male", 0), cat.get("female", 0)

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Concept mentions", "Scientist mentions"),
        horizontal_spacing=0.12,
    )
    for col, kind in ((1, "concept"), (2, "scientist")):
        ks4_totals = []
        ks5_totals = []
        hover_g = []
        hover_a = []
        for sb in subjects:
            mc, fc = gender_split(totals_g, sb, kind)
            ms, fs = gender_split(totals_a, sb, kind)
            ks4_totals.append(mc + fc)
            ks5_totals.append(ms + fs)
            hover_g.append(f"KS4 {kind} mentions<br>Total: {mc + fc}<br>Men: {mc}<br>Women: {fc}")
            hover_a.append(f"KS5 {kind} mentions<br>Total: {ms + fs}<br>Men: {ms}<br>Women: {fs}")
        fig.add_trace(
            go.Bar(
                y=y_labels,
                x=ks4_totals,
                name="KS4 (GCSE / SQ5)",
                orientation="h",
                marker_color=pf.PURPLE,
                hovertext=hover_g,
                hoverinfo="text",
                showlegend=col == 1,
            ),
            row=1,
            col=col,
        )
        fig.add_trace(
            go.Bar(
                y=y_labels,
                x=ks5_totals,
                name="KS5 (A-Level / SQ Adv + Highers)",
                orientation="h",
                marker_color=pf.PURPLE_LIGHT,
                hovertext=hover_a,
                hoverinfo="text",
                showlegend=col == 1,
            ),
            row=1,
            col=col,
        )
    fig.update_layout(
        barmode="group",
        title=f"Subject breakdown - KS4 vs KS5 ({scope_label(STAGE_BOTH, board_sel)})",
        height=max(420, 60 * len(subjects) + 140),
        margin=dict(t=80, l=140),
    )
    fig.update_xaxes(title_text="Mentions")
    return fig


def gender_comparison_figure(board_sel):
    male_g, female_g = unique_gender_counts(merge_scientists("ks4", board_sel))
    male_a, female_a = unique_gender_counts(merge_scientists("ks5", board_sel))
    return grouped_stage_bars(
        ["Women", "Men"],
        [female_g, male_g],
        [female_a, male_a],
        [f"KS4 women<br>Count: {female_g}", f"KS4 men<br>Count: {male_g}"],
        [f"KS5 women<br>Count: {female_a}", f"KS5 men<br>Count: {male_a}"],
        ["female", "male"],
        f"Unique named scientists by gender - KS4 vs KS5 ({scope_label(STAGE_BOTH, board_sel)})",
        "Unique scientists",
        420,
    )


def mention_type_comparison_figure(board_sel):
    concept_g, scientist_g = mention_type_counts("ks4", board_sel)
    concept_a, scientist_a = mention_type_counts("ks5", board_sel)
    return grouped_stage_bars(
        ["Scientist", "Concept"],
        [scientist_g, concept_g],
        [scientist_a, concept_a],
        [f"KS4 scientist mentions<br>Count: {scientist_g}", f"KS4 concept mentions<br>Count: {concept_g}"],
        [f"KS5 scientist mentions<br>Count: {scientist_a}", f"KS5 concept mentions<br>Count: {concept_a}"],
        ["scientist", "concept"],
        f"Mention type - KS4 vs KS5 ({scope_label(STAGE_BOTH, board_sel)})",
        "Mentions",
        420,
    )


def composition_comparison_figure(board_sel, dimension, title_prefix):
    scientists_g = merge_scientists("ks4", board_sel)
    scientists_a = merge_scientists("ks5", board_sel)
    counts_g = Counter(str(v.get(dimension, "unknown")).lower() for v in scientists_g.values())
    counts_a = Counter(str(v.get(dimension, "unknown")).lower() for v in scientists_a.values())
    all_keys = sorted(set(counts_g) | set(counts_a), key=lambda k: counts_g.get(k, 0) + counts_a.get(k, 0), reverse=True)
    total_g = sum(counts_g.values()) or 1
    total_a = sum(counts_a.values()) or 1
    y_labels = [prettify(k) for k in all_keys]
    hover_g = [
        f"KS4 - {prettify(k)}<br>Scientists: {counts_g.get(k, 0)}<br>Share: {counts_g.get(k, 0) / total_g * 100:.1f}%"
        for k in all_keys
    ]
    hover_a = [
        f"KS5 - {prettify(k)}<br>Scientists: {counts_a.get(k, 0)}<br>Share: {counts_a.get(k, 0) / total_a * 100:.1f}%"
        for k in all_keys
    ]
    return grouped_stage_bars(
        y_labels,
        [counts_g.get(k, 0) / total_g * 100 for k in all_keys],
        [counts_a.get(k, 0) / total_a * 100 for k in all_keys],
        hover_g,
        hover_a,
        all_keys,
        f"{title_prefix} - KS4 vs KS5 (% of unique scientists, {scope_label(STAGE_BOTH, board_sel)})",
        "Percentage (%)",
        max(420, 42 * len(all_keys) + 140),
    )


def build_figure(stage_key, board_sel, view_name):
    if stage_key == STAGE_BOTH:
        if view_name == "Subject breakdown (by gender)":
            return subject_comparison_figure(board_sel)
        if view_name == "Unique scientists by gender":
            return gender_comparison_figure(board_sel)
        if view_name == "Concept vs scientist mentions":
            return mention_type_comparison_figure(board_sel)
        if view_name == "Scientists by region":
            return composition_comparison_figure(board_sel, "region", "Regional representation")
        if view_name == "Scientists by nationality":
            return composition_comparison_figure(board_sel, "nationality", "Nationality")
        raise ValueError(view_name)
    if view_name == "Subject breakdown (by gender)":
        return subject_figure(stage_key, board_sel)
    if view_name == "Unique scientists by gender":
        return gender_figure(stage_key, board_sel)
    if view_name == "Concept vs scientist mentions":
        return mention_type_figure(stage_key, board_sel)
    if view_name == "Scientists by region":
        return region_figure(stage_key, board_sel)
    if view_name == "Scientists by nationality":
        return nationality_figure(stage_key, board_sel)
    raise ValueError(view_name)


def summary_html(stage_key, board_sel, view_name):
    scientists = scientists_for_filters(stage_key, board_sel)
    board_label = scope_label(stage_key, board_sel)
    if stage_key == STAGE_BOTH:
        n4 = len(merge_scientists("ks4", board_sel))
        n5 = len(merge_scientists("ks5", board_sel))
        combined = len(scientists)
        return (
            f"<div style='padding:10px 14px; background:#f0f6fc; border-radius:8px; margin-bottom:10px;'>"
            f"<strong>{board_label}</strong> | Comparing KS4 and KS5 | "
            f"KS4 unique: {n4} | KS5 unique: {n5} | Combined unique: {combined} | View: {view_name}"
            f"</div>"
        )
    total = len(scientists)
    women = sum(1 for s in scientists.values() if str(s.get("gender", "")).lower() == "female")
    men = sum(1 for s in scientists.values() if str(s.get("gender", "")).lower() == "male")
    pct_women = women / total * 100 if total else 0
    return (
        f"<div style='padding:10px 14px; background:#f0f6fc; border-radius:8px; margin-bottom:10px;'>"
        f"<strong>{board_label}</strong> | {total} unique scientists | "
        f"{women} women ({pct_women:.1f}%) | {men} men | View: {view_name}"
        f"</div>"
    )


In [3]:
stage_dropdown = widgets.Dropdown(
    options=list(KEY_STAGE_OPTIONS.keys()),
    value=list(KEY_STAGE_OPTIONS.keys())[0],
    description="Key stage:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

board_dropdown = widgets.Dropdown(
    options=[ALL_BOARDS],
    value=ALL_BOARDS,
    description="Exam board:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

view_dropdown = widgets.Dropdown(
    options=VIEW_OPTIONS,
    value=VIEW_OPTIONS[0],
    description="Chart view:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

demo_dimension_dropdown = widgets.Dropdown(
    options=list(DEMOGRAPHIC_OPTIONS.keys()),
    value="Gender",
    description="Explore by:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

demo_value_dropdown = widgets.Dropdown(
    description="Group:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

summary_out = widgets.Output()
names_out = widgets.Output()
figure_widget = go.FigureWidget()


def refresh_board_options(*_):
    stage_key = KEY_STAGE_OPTIONS[stage_dropdown.value]
    boards = board_options(stage_key)
    previous = board_dropdown.value
    board_dropdown.options = [(format_board_option(stage_key, b), b) for b in boards]
    if previous in boards:
        board_dropdown.value = previous
    else:
        board_dropdown.value = ALL_BOARDS


def refresh_demo_values(*_):
    stage_key = KEY_STAGE_OPTIONS[stage_dropdown.value]
    scientists = scientists_for_filters(stage_key, board_dropdown.value)
    dimension = DEMOGRAPHIC_OPTIONS[demo_dimension_dropdown.value]
    values = demographic_values(scientists, dimension)
    demo_value_dropdown.options = [(prettify(v), v) for v in values]
    if values:
        demo_value_dropdown.value = values[0]


def show_names_for_selection(dimension=None, raw_value=None, extra_title="", stage_for_names=None):
    stage_key = KEY_STAGE_OPTIONS[stage_dropdown.value]
    board_sel = board_dropdown.value
    if stage_for_names in ("ks4", "ks5"):
        scientists = merge_scientists(stage_for_names, board_sel)
        stage_note = " - KS4" if stage_for_names == "ks4" else " - KS5"
    else:
        scientists = scientists_for_filters(stage_key, board_sel)
        stage_note = ""
    kwargs = {}
    if dimension and raw_value is not None:
        kwargs[dimension] = raw_value
    rows = filter_scientists(scientists, **kwargs)
    if dimension and raw_value is not None:
        title = f"{prettify(raw_value)} ({prettify(dimension)}){stage_note}{extra_title}"
    else:
        title = f"All scientists - {scope_label(stage_key, board_sel)}{stage_note}{extra_title}"
    with names_out:
        names_out.clear_output(wait=True)
        display(HTML(names_html(rows, title)))


def attach_click_handlers(*_):
    for trace in figure_widget.data:
        trace.on_click(on_chart_click)


def refresh_chart(*_):
    if not board_dropdown.value:
        return
    stage_key = KEY_STAGE_OPTIONS[stage_dropdown.value]
    view_name = view_dropdown.value
    with summary_out:
        summary_out.clear_output(wait=True)
        display(HTML(summary_html(stage_key, board_dropdown.value, view_name)))
    fig = build_figure(stage_key, board_dropdown.value, view_name)
    figure_widget.data = []
    figure_widget.layout = fig.layout
    for trace in fig.data:
        figure_widget.add_trace(trace)
    attach_click_handlers()
    refresh_demo_values()
    show_names_for_selection()


def on_chart_click(trace, points, selector):
    if not points.point_inds:
        return
    idx = points.point_inds[0]
    view_name = view_dropdown.value
    stage_key = KEY_STAGE_OPTIONS[stage_dropdown.value]
    comparing = stage_key == STAGE_BOTH
    clicked_stage = None
    if comparing and str(trace.name or "").startswith("KS4"):
        clicked_stage = "ks4"
    elif comparing and str(trace.name or "").startswith("KS5"):
        clicked_stage = "ks5"
    extra = " (clicked on chart)"
    if view_name == "Unique scientists by gender":
        raw = trace.customdata[idx] if trace.customdata is not None else ["male", "female"][idx]
        show_names_for_selection("gender", raw, extra, stage_for_names=clicked_stage)
        demo_dimension_dropdown.value = "Gender"
        demo_value_dropdown.value = raw
    elif view_name in ("Scientists by region", "Scientists by nationality"):
        raw = trace.customdata[idx]
        dimension = "region" if view_name == "Scientists by region" else "nationality"
        show_names_for_selection(dimension, raw, extra, stage_for_names=clicked_stage)
        demo_dimension_dropdown.value = "Region" if dimension == "region" else "Nationality"
        demo_value_dropdown.value = raw
    elif view_name == "Concept vs scientist mentions":
        raw = trace.customdata[idx] if trace.customdata is not None else ["concept", "scientist"][idx]
        with names_out:
            names_out.clear_output(wait=True)
            display(HTML(
                f"<p><em>Mention-type charts count <strong>all mentions</strong>, not unique people. "
                f"Selected: <strong>{prettify(raw)}</strong>. Use the demographic explorer below to list individuals.</em></p>"
            ))
    elif view_name == "Subject breakdown (by gender)":
        with names_out:
            names_out.clear_output(wait=True)
            display(HTML(
                "<p><em>Subject charts count mentions, not unique people. "
                "Use the demographic explorer below to list individuals.</em></p>"
            ))


def on_demo_change(*_):
    dimension = DEMOGRAPHIC_OPTIONS[demo_dimension_dropdown.value]
    raw_value = demo_value_dropdown.value
    show_names_for_selection(dimension, raw_value)


stage_dropdown.observe(refresh_board_options, names="value")
stage_dropdown.observe(refresh_chart, names="value")
board_dropdown.observe(refresh_chart, names="value")
board_dropdown.observe(refresh_demo_values, names="value")
view_dropdown.observe(refresh_chart, names="value")
demo_dimension_dropdown.observe(refresh_demo_values, names="value")
demo_dimension_dropdown.observe(on_demo_change, names="value")
demo_value_dropdown.observe(on_demo_change, names="value")

refresh_board_options()
refresh_chart()

controls = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:8px'>Filters</h3>"),
    widgets.HBox([stage_dropdown, board_dropdown, view_dropdown], layout=widgets.Layout(flex_wrap="wrap")),
    widgets.HTML("<h3 style='margin:18px 0 8px'>Demographic explorer</h3><p style='margin-top:0;color:#555'>Pick a group to list every named scientist, or click a chart segment above.</p>"),
    widgets.HBox([demo_dimension_dropdown, demo_value_dropdown], layout=widgets.Layout(flex_wrap="wrap")),
])

dashboard = widgets.VBox([
    controls,
    summary_out,
    figure_widget,
    widgets.HTML("<h3 style='margin:18px 0 8px'>Scientists in selected group</h3>"),
    names_out,
])

display(dashboard)
